# Tutorial 2: one-loop integrals with numerators

**What you will learn**
1. how to write a one-loop integral as strings and let `loop()` reduce it to $A_0, B_0, C_0, D_0$
2. how to check a tensor reduction by hand (the rank-one bubble)
3. triangles and boxes: exact symbols, 30-digit numbers, closed forms in dilogarithms
4. soft and collinear poles ($1/\epsilon^2$)

This is what Package-X does. The conventions are the same, so every number here can be compared with Package-X or LoopTools.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))     # use the feynsage of this repository (not needed after ./install.sh)
from feynsage import *
%display latex

## Step 1: a scalar bubble from strings

`loop(numerator, propagator, propagator, ..., kin=...)`. Each propagator is `[momentum, mass]`. `kin` gives the scalar products of the external momenta.

In [2]:
loop("1", ["l", "m1"], ["l + p", "m2"], kin={"p^2": "s"})

1 :  -1/2*(m1^2 - m2^2 + s)*log(m1^2/m2^2)/s + 1/eps + DiscB(s, m1, m2) + log(mu^2/m2^2) + 2

This is $B_0(s; m_1, m_2)$ written out. The term with $\log(m_1^2/m_2^2)$ appears because the masses differ.

## Step 2: a vector numerator

With $\ell^\mu$ upstairs the only vector the answer can point along is $p^\mu$, so the integral is $p^\mu B_1$.

In [3]:
r1 = loop("l^mu", ["l", "m1"], ["l + p", "m2"], kin={"p^2": "s"})
r1

p^mu :  1/2*m1^2*(1/eps + log(mu^2/m1^2) + 1)/s - 1/2*m2^2*(1/eps + log(mu^2/m2^2) + 1)/s + 1/4*(m1^2 - m2^2 + s)*((m1^2 - m2^2 + s)*log(m1^2/m2^2)/s - 2/eps - 2*DiscB(s, m1, m2) - 2*log(mu^2/m2^2) - 4)/s

**Check by hand.** Contract with $p_\mu$ and use $2\ell\cdot p = D_2 - D_1 - (s + m_1^2 - m_2^2)$, where $D_1 = \ell^2 - m_1^2$ and $D_2 = (\ell+p)^2 - m_2^2$. Each $D_i$ cancels a propagator and leaves a tadpole:
$$B_1 = \frac{1}{2s}\Big[A_0(m_1) - A_0(m_2) - (s + m_1^2 - m_2^2)\,B_0(s; m_1, m_2)\Big] .$$
`PVB(0, 1, s, m1, m2)` is Package-X's `PVB[0, 1, ...]`, that is $B_1$. Compare at a point above threshold.

In [4]:
sv, a, b = 10, 1, 2
hand = (A0(a) - A0(b) - (sv + a^2 - b^2)*B0(sv, a, b))/(2*sv)
pv = PVB(0, 1, sv, a, b)
print("poles:        ", uv_part(hand), uv_part(pv))
print("finite parts: ", finite_part(hand, 1).n(), "   ", finite_part(pv, 1).n())

poles:         -1/2 -1/2
finite parts:  -0.184578706662452 - 0.329867228626928*I     -0.184578706662452 - 0.329867228626928*I


## Step 3: rank two, with the explanation switched on

$\int \ell^\mu \ell^\nu/(D_1 D_2) = g^{\mu\nu} B_{00} + p^\mu p^\nu B_{11}$. With `explain=True` the function says what each output means.

In [5]:
loop("l^mu l^nu", ["l", "m1"], ["l + p", "m2"], kin={"p^2": "s"}, explain=True)

loop(): a one-loop integral reduced to scalar functions.
  Normalisation (Package-X / LoopTools): mu^(2 eps) e^(eps gamma_E) Int d^d l/(i pi^(d/2)),
  d = 4 - 2 eps, propagators 1/(q^2 - m^2 + i0) [Minkowski] or, with euclidean=True,
  Int d^d l/pi^(d/2) with 1/(q^2 + m^2) [Euclidean; the functions are then the Minkowski
  ones at p_M^2 = -p_E^2 times (-1)^N].
  Output: one line per tensor structure (g^{mu nu}, p^mu, ... or 1 for a scalar numerator),
  each with its coefficient expanded to eps^0.  The pieces are
    1/eps^2, 1/eps  poles: UV, and IR (soft/collinear) from divergent C0, D0 written out,
    A0, B0      written out: logs, LogM(z) = log(z - i0), DiscB(s, m1, m2) (see B0?),
    C0(...), D0(...)  IR-finite ones stay symbols; .n() gives > 30 digits, explicit() the dilogarithms,
    mu          the renormalisation scale.
  .masters() gives the exact coefficients in d before expanding; .coefficients() a dict.


g^{mu nu} :  1/12*(m1^2 - m2^2 + s)*m1^2*(1/eps + log(mu^2/m1^2) + 1)/s - 1/12*(m1^2 - m2^2 - s)*m2^2*(1/eps + log(mu^2/m2^2) + 1)/s + 1/18*(m1^2 - m2^2 + s)*m1^2/s - 1/18*(m1^2 - m2^2 - s)*m2^2/s + 1/24*(m1^4 - 2*m1^2*m2^2 + m2^4 - 2*m1^2*s - 2*m2^2*s + s^2)*((m1^2 - m2^2 + s)*log(m1^2/m2^2)/s - 2/eps - 2*DiscB(s, m1, m2) - 2*log(mu^2/m2^2) - 4)/s - 1/18*(m1^4 - 2*m1^2*m2^2 + m2^4 - 2*m1^2*s - 2*m2^2*s + s^2)/s
p^mu p^nu :  -1/3*(m1^2 - m2^2 + s)*m1^2*(1/eps + log(mu^2/m1^2) + 1)/s^2 + 1/3*(m1^2 - m2^2 + 2*s)*m2^2*(1/eps + log(mu^2/m2^2) + 1)/s^2 - 1/18*(m1^2 - m2^2 + s)*m1^2/s^2 + 1/18*(m1^2 - m2^2 - s)*m2^2/s^2 - 1/6*(m1^4 - 2*m1^2*m2^2 + m2^4 + m1^2*s - 2*m2^2*s + s^2)*((m1^2 - m2^2 + s)*log(m1^2/m2^2)/s - 2/eps - 2*DiscB(s, m1, m2) - 2*log(mu^2/m2^2) - 4)/s^2 + 1/18*(m1^4 - 2*m1^2*m2^2 + m2^4 - 2*m1^2*s - 2*m2^2*s + s^2)/s^2

## Step 4: contracted numerators

Numerators like $\ell^2$ or $\ell\cdot p$ are written the same way. Since $\ell^2 = D_1 + m_1^2$, the integral with $\ell^2$ upstairs must be $A_0(m_2) + m_1^2 B_0$. Check it:

In [6]:
s, m1, m2 = var('s m1 m2')
l2 = SR(loop("l^2", ["l", "m1"], ["l + p", "m2"], kin={"p^2": "s"})["1"])
(l2 - (A0(m2) + m1^2*B0(s, m1, m2))).simplify_full()

0

`loop()` returns a `LoopResult`. Index it with the tensor structure, here `"1"` for the scalar part, to get the plain Sage expression.

## Step 5: a triangle

`C0(s1, s12, s2, m0, m1, m2)` with propagators $(\ell, m_0)$, $(\ell + p_1, m_1)$, $(\ell + p_2, m_2)$, $s_1 = p_1^2$, $s_2 = p_2^2$, $s_{12} = (p_1 - p_2)^2$ (Package-X order). A finite triangle stays a symbol until you ask for a number.

In [7]:
c = C0(1, 2, 3, 1, 2, 3)
c

C0(1, 2, 3, 1, 2, 3)

In [8]:
print("30 digits:            ", c.n(digits=30))
print("numerical integration:", c0_numeric(1, 2, 3, 1, 2, 3))

30 digits:             -0.142398839895237347030644540980
numerical integration: (-0.1423988398952359+0j)


`c0_numeric` integrates the Feynman-parameter form directly: a second, independent way. They agree to the 13 digits that the numerical integration can give.

`explicit()` writes the triangle in logarithms and dilogarithms of exact algebraic numbers:

In [9]:
e = explicit(c)
print(len(str(e)), "characters;  value:", e.n(digits=20))

1206 characters;  value: -0.14239883989523734703 - 1.1978854817838860011e-21*I


## Step 6: a box

`D0(s1, s2, s3, s4, s12, s23, m0, m1, m2, m3)` in the LoopTools order. Again a symbol, its number and an independent integration. (Pick a point away from thresholds for `d0_numeric`: right on a threshold such as $s_4 = (m_0 + m_3)^2$ the integrand has a soft edge and the simple numerical grid becomes inaccurate. The closed form is fine there.)

In [10]:
d = D0(-1, -2, -3, -4, -5, -6, 1, 2, 3, 4)
print(d.n(digits=25))
print(d0_numeric(-1, -2, -3, -4, -5, -6, 1, 2, 3, 4))

0.003369962787211075225506485


(0.0033699627872110765+0j)


## Step 7: soft and collinear poles

A massless triangle with two light-like legs has a $1/\epsilon^2$: one $1/\epsilon$ from soft and one from collinear loop momenta. Here $p_1^2 = p_2^2 = 0$ and $s_{12} = s$.

In [11]:
s = var('s')
cir = C0(0, s, 0, 0, 0, 0)
cir

-1/12*pi^2/s + 1/2*LogM(-s)^2/s - LogM(-s)*log(mu^2)/s + 1/2*log(mu^2)^2/s - (LogM(-s)/s - log(mu^2)/s)/eps + 1/(eps^2*s)

In [12]:
pole_parts(cir)

(1/s, -LogM(-s)/s + log(mu^2)/s)

Compare with the lecture note (Section 14, the massless triangle in Euclidean space, $Q^2 = -s$):
$$\frac{e^{-\gamma_E\epsilon}}{Q^2}\Big[\frac{1}{\epsilon^2} - \frac{\log Q^2}{\epsilon} + \frac12\log^2 Q^2 - \frac{\pi^2}{12}\Big] .$$
With $\log Q^2 = \log(-s - i0)$ every term is there with the same constant $-\pi^2/12$. Two small differences:
- the factor $e^{\epsilon\gamma_E}$ in the normalisation removes the $e^{-\gamma_E\epsilon}$ in front
- the overall sign is opposite, because $1/Q^2 = -1/s$. The Wick rotation gives $d^d\ell_M = i\,d^d\ell_E$ (cancelled by the $1/i$) and $\ell_M^2 = -\ell_E^2$ in each of the three propagators, so $C_0 = (-1)^3\times$ (Euclidean triangle)

## Exercises

**1.** For equal masses the substitution $\ell \to -\ell - p$ swaps the two propagators. Use it to show $B_1 = -B_0/2$ and check it with `PVB`.
<details><summary>Show a solution</summary>

```python
s, m = var('s m')
(PVB(0, 1, s, m, m) + B0(s, m, m)/2).simplify_full()
```
</details>

**2.** A triangle does not change when you relabel its legs. Check that $C_0(s_1, s_{12}, s_2; m_0, m_1, m_2) = C_0(s_2, s_{12}, s_1; m_0, m_2, m_1)$ at a point of your choice.
<details><summary>Show a solution</summary>

```python
print(C0(1, 2, 3, 1, 2, 3).n(digits=25))
print(C0(3, 2, 1, 1, 3, 2).n(digits=25))
```
</details>

**3.** Reduce the rank-three bubble $\int \ell^\mu\ell^\nu\ell^\rho/(D_1 D_2)$ with equal masses. Which tensor structures appear?
<details><summary>Show a solution</summary>

```python
loop("l^mu l^nu l^rho", ["l", "m"], ["l + p", "m"], kin={"p^2": "s"})
```
</details>